# 🎬 10Eros LTX 2.3 — Image-to-Video en Google Colab (versión GGUF)

Copia independiente del Space `signsur4739379373/LTX-2.3-10Eros`, optimizada para el disco de Colab gratuito:

| Optimización | Ahorro |
|---|---|
| Checkpoint fp8 (29.2 GB) → **GGUF Q4_K_S (13.2 GB)** + VAEs sueltos (1.8 GB) | **~14 GB** |
| Prompt enhancer desactivado por defecto (modelo sulphur + llama.cpp) | **~14 GB** |
| Limpieza de cachés pip/apt/git | ~5-10 GB |

Además puedes **añadir LoRAs de CivitAI** (Celda 5): marcas con **checkboxes** cuáles instalar (son opcionales) y eliges por slot en la UI — con un dropdown, exclusivo de las custom — si usar la LoRA original o una de tus custom marcadas.

---
**Instrucciones:**
1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`
2. Ejecuta las celdas **en orden** (la 5 de CivitAI es opcional)
3. La celda 7 valida tu GPU/disco y desbloquea los modelos ejecutables en tu entorno
4. La celda 8 lanza la app **en segundo plano** y muestra el enlace `gradio.live` — la UI no se incrusta en el notebook; logs en `/content/app.log`

## 📊 Celda 1 — Verificar GPU y CUDA

In [ ]:
# ─── Verificación de GPU ───────────────────────────────────────────────────
import subprocess, sys, os

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("❌ No se detectó GPU. Ve a Entorno de ejecución → Cambiar tipo → GPU")
print(result.stdout)

# Detectar versión de CUDA del driver
cuda_ver_line = [l for l in result.stdout.splitlines() if 'CUDA Version' in l]
if cuda_ver_line:
    cuda_str = cuda_ver_line[0].split('CUDA Version:')[-1].strip().split()[0]
    cuda_major, cuda_minor = int(cuda_str.split('.')[0]), int(cuda_str.split('.')[1])
    print(f"\n✅ CUDA detectado: {cuda_str}")
    if cuda_major >= 12 and cuda_minor >= 8:
        TORCH_CUDA_TAG = 'cu128'
    elif cuda_major >= 12 and cuda_minor >= 4:
        TORCH_CUDA_TAG = 'cu124'
    elif cuda_major >= 12:
        TORCH_CUDA_TAG = 'cu121'
    else:
        TORCH_CUDA_TAG = 'cu118'
    print(f"📦 Se usará índice PyTorch: {TORCH_CUDA_TAG}")
else:
    TORCH_CUDA_TAG = 'cu121'
    print("⚠️  No se pudo determinar CUDA, usando cu121 como fallback")

os.environ['TORCH_CUDA_TAG'] = TORCH_CUDA_TAG

# Exportar GPU info para Celda 7 (evita segunda llamada a nvidia-smi)
try:
    _g = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
        capture_output=True, text=True, check=True).stdout.strip().splitlines()[0]
    _gpu_name, _vram_mb = (s.strip() for s in _g.split(','))
    os.environ['GPU_NAME'] = _gpu_name
    os.environ['GPU_VRAM_MB'] = _vram_mb
except Exception:
    pass
print("✅ GPU lista para usar")

## 🔑 Celda 2 — Tokens (HuggingFace + CivitAI)

> - **HF_TOKEN**: créalo en https://huggingface.co/settings/tokens (tipo `read`)
> - **CIVITAI_TOKEN**: créalo en https://civitai.com/user/account → API Keys (necesario para descargar la mayoría de LoRAs de CivitAI)

In [ ]:
# ─── Configuración de tokens ───────────────────────────────────────────────
HF_TOKEN = ""        #@param {type:"string"}
CIVITAI_TOKEN = ""   #@param {type:"string"}

import os, subprocess

if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN.strip()
    print("✅ HF_TOKEN configurado (repos privados/gated + sin rate-limit)")
else:
    print("ℹ️  Sin HF_TOKEN — las descargas públicas funcionarán igual")

if CIVITAI_TOKEN.strip():
    os.environ['CIVITAI_TOKEN'] = CIVITAI_TOKEN.strip()
    print("✅ CIVITAI_TOKEN configurado")
else:
    print("ℹ️  Sin CIVITAI_TOKEN — muchas descargas de CivitAI fallarán con 401")

# hf_transfer: descarga paralela en Rust — esto SÍ acelera las descargas de HF
subprocess.run(['pip', 'install', '-q', 'hf_transfer'], check=False)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print("🚀 hf_transfer activado — descargas de HuggingFace aceleradas")

## ⬇️ Celda 3 — Descargar código e Instalar Dependencias
> El código (`app.py` + workflow MSR) se descarga de **github.com/aiRabbit0/SpaceLTX-GoogleColab**
> ⚠️ Tarda ~5-10 minutos la primera vez. Al final purga cachés de pip/apt para ahorrar disco.

In [ ]:
# ─── Descargar código desde GitHub e instalar dependencias ────────────────
import os, subprocess, sys

CODE_RAW  = 'https://raw.githubusercontent.com/aiRabbit0/SpaceLTX-GoogleColab/main'
SPACE_DIR = '/content/LTX-10Eros'
TORCH_CUDA_TAG = os.environ.get('TORCH_CUDA_TAG', 'cu121')

def run(cmd, **kw):
    print(f"$ {cmd}")
    proc = subprocess.run(cmd, shell=True, text=True, **kw)
    return proc.returncode

# El código vive en el repositorio de GitHub (no en el Space de HF). Para usar
# un fork propio, cambia CODE_RAW por la URL raw de ese fork.
# Renombres: app_space.py → app.py | workflow_runexx.json → runexx_msr_workflow.json
os.makedirs(SPACE_DIR, exist_ok=True)
CODE_FILES = {
    'app_space.py':         'app.py',
    'workflow_runexx.json': 'runexx_msr_workflow.json',
}
print(f"📥 Descargando código desde {CODE_RAW} ...")
for src, dst in CODE_FILES.items():
    dest = os.path.join(SPACE_DIR, dst)
    rc = run(f'wget -q "{CODE_RAW}/{src}" -O "{dest}"')
    if rc != 0 or os.path.getsize(dest) == 0:
        raise RuntimeError(f'❌ No se pudo descargar {src} desde GitHub. '
                           '¿El repo es público y el archivo existe en main?')
    print(f"   ✅ {src} → {dst} ({os.path.getsize(dest)/1024:.0f} KB)")

print("\n📦 Instalando PyTorch 2.8.0...")
torch_index = f'https://download.pytorch.org/whl/{TORCH_CUDA_TAG}'
rc = run(f'pip install -q torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --extra-index-url {torch_index}')
if rc != 0:
    print("⚠️  torch 2.8.0 no disponible, instalando 2.4.0 como fallback...")
    run('pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --extra-index-url https://download.pytorch.org/whl/cu121')

print("\n📦 Instalando resto de dependencias...")
deps = [
    'torchsde', 'gradio>=5,<6', 'huggingface_hub>=0.34.0', 'transformers>=4.48.0',
    'accelerate>=0.26.0', 'safetensors', 'einops', 'scipy', 'numpy', 'pillow',
    'opencv-python-headless', 'mediapipe', 'av', 'kornia<0.8.0', 'psutil',
    'websocket-client', 'gguf', 'sentencepiece', 'protobuf', 'spandrel',
    'soundfile', 'imageio-ffmpeg', 'rotary_embedding_torch',
]
run('pip install -q ' + ' '.join(f'"{d}"' for d in deps))

# ─── Limpieza de cachés (ahorro de disco) ──────────────────────────────────
print("\n🧹 Purgando cachés de pip y apt...")
run('pip cache purge')
run('apt-get clean')
run('rm -rf /root/.cache/pip')

import shutil
du = shutil.disk_usage('/content')
print(f"\n💽 Disco libre: {du.free/1024**3:.1f} GB")
print("✅ Código descargado, dependencias instaladas y cachés purgadas")

## 🩹 Celda 4 — Parchear módulo `spaces` (ZeroGPU → Colab)

> Opcional: la Celda 8 lanza la app en un subproceso con su propio mock de `spaces`. Esta celda solo hace falta si vas a importar `app.py` a mano en el kernel.

In [ ]:
# ─── Mock del módulo 'spaces' (OPCIONAL) ────────────────────────────────
# La Celda 8 lanza la app como subproceso con su propio mock; esta celda
# solo hace falta si importas app.py directamente en el kernel (desarrollo).
import sys, types, os

mock_spaces = types.ModuleType('spaces')

def _gpu_decorator(*args, **kwargs):
    if len(args) == 1 and callable(args[0]) and not kwargs:
        return args[0]
    def decorator(fn):
        return fn
    return decorator

mock_spaces.GPU = _gpu_decorator
mock_spaces.ZeroGPU = _gpu_decorator
sys.modules['spaces'] = mock_spaces

SPACE_DIR = '/content/LTX-10Eros'
if SPACE_DIR not in sys.path:
    sys.path.insert(0, SPACE_DIR)
os.chdir(SPACE_DIR)
print("ℹ️  Mock de 'spaces' aplicado al kernel (opcional — la Celda 8 usa el suyo propio)")

## 🎨 Celda 5 — LoRAs personalizadas de CivitAI (opcional)

Esta celda descarga tus LoRAs a `/content/custom_loras/` y al final muestra **checkboxes para marcar cuáles instalar** (igual que las LoRAs originales de la Celda 8 — son opcionales). Las **desmarcadas no se instalan y no aparecen en la app**. El **dropdown** de cada slot en la UI es exclusivo de las custom: ofrece la original del slot + solo tus custom marcadas + `(none)`.

### 📝 Cómo agregar una LoRA custom, paso a paso

1. Busca la LoRA en [civitai.com](https://civitai.com) y comprueba en su ficha que el **base model sea LTX 2.3** (las de SDXL/Flux/Wan no funcionan).
2. Copia el enlace. Sirven estos formatos:
   - URL completa con versión: `https://civitai.com/models/123456?modelVersionId=789012` ✅ (recomendado)
   - URL del modelo a secas: `https://civitai.com/models/123456` (se usa su versión más reciente)
   - Solo el ID de versión: `789012`
3. Pégalo en el campo `CUSTOM_LORAS_LINKS` del formulario (varios enlaces separados por **comas**) o añade líneas a la lista `CUSTOM_LORAS` del código — la lista no tiene límite: agrega tantas líneas como quieras.
4. Si la descarga da **401**, la LoRA requiere cuenta: configura `CIVITAI_TOKEN` en la Celda 2. Si la celda avisa de **HTML/gated**, abre el enlace en el navegador y acepta los términos del modelo primero.
5. Ejecuta la celda: verás los **metadatos** (nombre real, versión, base model, trigger words, tamaño), la descarga con progreso y al final los **checkboxes de selección** — marcar/desmarcar se guarda al instante (`selected.json`), sin re-ejecutar la celda. Re-ejecutar es seguro: lo ya descargado no se repite.
6. Tras lanzar la app (Celda 8) — que instala **solo las marcadas** y retira las desmarcadas: en el **dropdown del slot** que prefieras elige `custom: <nombre>`, sube el slider de ese slot (video y/o audio) y usa las **trigger words** en tu prompt. Si cambias la selección, re-ejecuta la Celda 8 (reinicia la app sin re-descargar nada).

> ℹ️ Los formularios de Colab no permiten añadir campos dinámicamente (botón "+"); por eso el campo de texto acepta N enlaces separados por comas y la lista de código es extensible a mano.

Las LoRAs **originales** de la app no se tocan aquí: se descargan según los checkboxes de la Celda 8 (por defecto, todas).

In [ ]:
# ─── Descarga de LoRAs custom desde CivitAI ───────────────────────────────
# Pega uno o VARIOS enlaces/IDs separados por comas en este campo del
# formulario (Colab no permite añadir campos dinámicamente, así que este
# campo acepta N enlaces de una vez):
CUSTOM_LORAS_LINKS = ""  #@param {type:"string"}

# ...o añade líneas a esta lista (sin límite; "" = ignorado):
CUSTOM_LORAS = [
    "",   # p.ej. https://civitai.com/models/123456?modelVersionId=789012
    "",   # p.ej. 789012 (ID de versión a secas)
    "",
]

import os, re, json, pathlib, requests

CUSTOM_LORAS += [s for s in re.split(r'[,\s]+', CUSTOM_LORAS_LINKS) if s]

CIVITAI_TOKEN = os.environ.get('CIVITAI_TOKEN', '')
STAGING = pathlib.Path('/content/custom_loras')
STAGING.mkdir(exist_ok=True)
MANIFEST_PATH = STAGING / 'mapping.json'

def resolve_version_id(ref: str):
    """Acepta URL completa de CivitAI o un ID numérico de versión."""
    ref = ref.strip()
    if not ref:
        return None
    if re.fullmatch(r'\d+', ref):
        return ref
    m = re.search(r'modelVersionId=(\d+)', ref)
    if m:
        return m.group(1)
    m = re.search(r'civitai\.com/models/(\d+)', ref)
    if m:
        # Sin modelVersionId: consultar la API para tomar la versión más reciente
        r = requests.get(f'https://civitai.com/api/v1/models/{m.group(1)}', timeout=30)
        r.raise_for_status()
        return str(r.json()['modelVersions'][0]['id'])
    raise ValueError(f'Referencia no reconocida: {ref}')

def fetch_version_info(version_id: str) -> dict:
    """Metadatos públicos de la versión: nombre, base model, trigger words, tamaño."""
    try:
        r = requests.get(f'https://civitai.com/api/v1/model-versions/{version_id}',
                         timeout=30, headers={'User-Agent': 'Mozilla/5.0'})
        r.raise_for_status()
        d = r.json()
        f0 = (d.get('files') or [{}])[0]
        return {
            'version_id':    str(d.get('id', version_id)),
            'name':          (d.get('model') or {}).get('name', ''),
            'version':       d.get('name', ''),
            'base_model':    d.get('baseModel', ''),
            'trained_words': d.get('trainedWords') or [],
            'size_mb':       round((f0.get('sizeKB') or 0) / 1024, 1),
            'url':           f"https://civitai.com/models/{d.get('modelId')}?modelVersionId={d.get('id')}",
        }
    except Exception as e:
        print(f'  ⚠️  Sin metadatos de CivitAI ({e}) — se continúa solo con el archivo')
        return {}

def print_lora_info(info: dict):
    if not info:
        return
    print(f'  📄 "{info["name"]}" — versión: {info["version"]}')
    print(f'     base model: {info["base_model"] or "?"} | tamaño: {info["size_mb"]:.0f} MB')
    if info['trained_words']:
        print(f'     trigger words: {", ".join(info["trained_words"][:8])}')
    if info['base_model'] and 'ltx' not in info['base_model'].lower():
        print(f'     ⚠️  base model "{info["base_model"]}" no parece LTX — probablemente incompatible')

def download_civitai(version_id: str) -> str:
    url = f'https://civitai.com/api/download/models/{version_id}'
    params = {'token': CIVITAI_TOKEN} if CIVITAI_TOKEN else {}
    with requests.get(url, params=params, stream=True, timeout=60,
                      headers={'User-Agent': 'Mozilla/5.0'}) as r:
        if r.status_code == 401:
            raise RuntimeError(
                'CivitAI devolvió 401: esta descarga requiere token. '
                'Configura CIVITAI_TOKEN en la Celda 2.')
        r.raise_for_status()
        # CivitAI responde 200 con una página HTML cuando el modelo es gated
        # (requiere login/aceptar términos) — sin este guard se guardaría el
        # HTML como .safetensors y ComfyUI fallaría al cargarlo.
        ctype = r.headers.get('Content-Type', '')
        if 'text/html' in ctype.lower():
            raise RuntimeError(
                'CivitAI devolvió HTML en vez del archivo (modelo gated o token '
                'inválido). Abre el enlace en el navegador y acepta los términos, '
                'o revisa tu CIVITAI_TOKEN.')
        cd = r.headers.get('Content-Disposition', '')
        m = re.search(r'filename="?([^";]+)', cd)
        fname = m.group(1) if m else f'civitai_{version_id}.safetensors'
        # sanitizar nombre
        fname = re.sub(r'[^\w.\- ]', '_', fname)
        if not fname.endswith('.safetensors'):
            fname += '.safetensors'
        dest = STAGING / fname
        if dest.exists():
            # Solo llegan aquí descargas completas: las parciales viven en .part
            print(f'  ✅ {fname} ya descargado')
            return fname
        # Descarga atómica: a .part y rename al terminar — una desconexión a
        # mitad no deja un .safetensors truncado que luego pase por válido.
        tmp = dest.with_suffix(dest.suffix + '.part')
        total = int(r.headers.get('Content-Length', 0))
        done = 0
        try:
            with open(tmp, 'wb') as f:
                for chunk in r.iter_content(chunk_size=1024*1024):
                    f.write(chunk)
                    done += len(chunk)
                    if total:
                        print(f'\r  ⬇️  {fname}: {done/1024**2:.0f}/{total/1024**2:.0f} MB', end='')
            print()
            if total and done != total:
                raise RuntimeError(f'descarga incompleta ({done}/{total} bytes)')
        except BaseException:
            tmp.unlink(missing_ok=True)
            raise
        tmp.rename(dest)
        return fname

# Manifest acumulativo keyed por archivo: {filename: {name, version, ...}}.
# La Celda 8 lo instala como manifest.json junto a las LoRAs para que los
# dropdowns de la app muestren el nombre real de CivitAI.
manifest = {}
if MANIFEST_PATH.exists():
    try:
        manifest = json.loads(MANIFEST_PATH.read_text())
    except Exception:
        manifest = {}

just_downloaded = set()  # referenciadas en esta ejecución -> marcadas por defecto
for ref in CUSTOM_LORAS:
    try:
        vid = resolve_version_id(ref)
    except Exception as e:
        print(f'❌ {ref!r}: {e}')
        continue
    if vid is None:
        continue
    print(f'🎨 Versión CivitAI {vid}')
    info = fetch_version_info(vid)
    print_lora_info(info)
    try:
        fname = download_civitai(vid)
    except Exception as e:
        print(f'❌ versión {vid}: {e}')
        continue
    manifest[fname] = info
    just_downloaded.add(fname)

MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))

# ── Selección por checkbox (como las LoRAs originales en la Celda 8) ──────
# Las DESMARCADAS no se copian al catálogo de la app, así que no aparecen en
# los dropdowns. La selección se guarda al instante en selected.json y la
# Celda 8 la aplica (instala marcadas, retira desmarcadas) en cada lanzamiento.
SELECTED_PATH = STAGING / 'selected.json'
available = sorted(STAGING.glob('*.safetensors'))

def _save_selection(names):
    SELECTED_PATH.write_text(json.dumps(sorted(names), indent=2, ensure_ascii=False))

if available:
    prev = None  # sin selección previa -> todas marcadas
    if SELECTED_PATH.exists():
        try:
            prev = set(json.loads(SELECTED_PATH.read_text()))
        except Exception:
            prev = None

    import ipywidgets as widgets
    from IPython.display import display as _display

    _boxes = []
    for f in available:
        name = (manifest.get(f.name) or {}).get('name') or f.stem
        checked = True if prev is None else (f.name in prev or f.name in just_downloaded)
        cb = widgets.Checkbox(
            value=checked, indent=False,
            description=f'{name}  ({f.name}, {f.stat().st_size/1024**2:.0f} MB)',
            layout=widgets.Layout(width='95%'),
        )
        cb._lora_file = f.name
        _boxes.append(cb)

    def _on_toggle(change):
        _save_selection({b._lora_file for b in _boxes if b.value})

    for b in _boxes:
        b.observe(_on_toggle, names='value')
    _save_selection({b._lora_file for b in _boxes if b.value})

    print('\n📋 Marca las LoRAs custom que quieres en la app (desmarcada = no aparece):')
    _display(widgets.VBox(_boxes))
    print('\nℹ️  Solo las marcadas aparecerán en los dropdowns de slot tras la Celda 8.')
else:
    _save_selection([])
    print('ℹ️  Sin LoRAs custom — los dropdowns solo mostrarán las originales')

## 📊 Celda 6 — Monitor de VRAM, Disco y Tiempo de Sesión
> Ejecuta esta celda **antes** de lanzar la app. Se actualiza cada 5 segundos.

In [ ]:
# ─── Monitor de VRAM + Disco + Tiempo de sesión ───────────────────────────
import torch, time, threading, datetime, shutil
from IPython.display import display, HTML
import ipywidgets as widgets

SESSION_START   = time.time()
COLAB_GPU_LIMIT = 5 * 3600   # ~5 horas (límite aproximado de cuenta gratuita)
_monitor_running = True

def fmt_time(seconds):
    h, rem = divmod(int(seconds), 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def color_for_pct(pct):
    return '#4CAF50' if pct < 60 else ('#FF9800' if pct < 80 else '#f44336')

def bar(pct, color):
    return (f'<div style="background:#333;border-radius:4px;height:12px;margin-bottom:10px">'
            f'<div style="background:{color};width:{min(pct,100):.1f}%;height:12px;'
            f'border-radius:4px;transition:width .4s"></div></div>')

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'
out = widgets.Output()
display(out)

def render_monitor():
    elapsed   = time.time() - SESSION_START
    remaining = max(0, COLAB_GPU_LIMIT - elapsed)
    t_pct     = min(100, (elapsed / COLAB_GPU_LIMIT) * 100)

    if torch.cuda.is_available():
        v_used  = torch.cuda.memory_allocated() / 1024**2
        v_resv  = torch.cuda.memory_reserved() / 1024**2
        v_total = torch.cuda.get_device_properties(0).total_memory / 1024**2
        v_pct   = (v_used / v_total) * 100 if v_total else 0
    else:
        v_used = v_resv = v_total = v_pct = 0

    du = shutil.disk_usage('/content')
    d_used  = (du.total - du.free) / 1024**3
    d_total = du.total / 1024**3
    d_pct   = (d_used / d_total) * 100 if d_total else 0

    vc, tc, dc = color_for_pct(v_pct), color_for_pct(t_pct), color_for_pct(d_pct)
    now_str = datetime.datetime.now().strftime('%H:%M:%S')

    html = f"""
<div style="font-family:monospace;background:#1e1e1e;color:#e0e0e0;padding:14px 18px;
            border-radius:10px;max-width:580px;border:1px solid #444;margin:4px 0">
  <div style="font-size:15px;font-weight:bold;margin-bottom:10px;color:#90CAF9">
    📊 Monitor — {now_str}</div>
  <div style="margin-bottom:8px">🖥️ <b>GPU:</b> {gpu_name}</div>
  <div style="margin-bottom:4px">💾 <b>VRAM:</b> {v_used:.0f} / {v_total:.0f} MB
    <span style="color:#aaa">(reservada {v_resv:.0f} MB)</span>
    <b style="color:{vc}">{v_pct:.1f}%</b></div>
  {bar(v_pct, vc)}
  <div style="margin-bottom:4px">💽 <b>Disco:</b> {d_used:.1f} / {d_total:.1f} GB
    <b style="color:{dc}">{d_pct:.1f}%</b></div>
  {bar(d_pct, dc)}
  <div style="margin-bottom:4px">⏱️ <b>Sesión:</b> {fmt_time(elapsed)}
    <b style="color:{tc}">{t_pct:.1f}% del límite</b></div>
  {bar(t_pct, tc)}
  <div>⏳ <b>Tiempo restante (aprox):</b>
    <b style="color:{tc}">{fmt_time(remaining)}</b>
    <span style="color:#aaa">(límite ~5h cuenta gratuita)</span></div>
</div>"""
    with out:
        out.clear_output(wait=True)
        display(HTML(html))

def update_monitor():
    while _monitor_running:
        try:
            render_monitor()
        except Exception:
            pass
        time.sleep(5)

# Primer render inmediato (síncrono) — así el panel aparece aunque las
# actualizaciones desde el hilo de fondo fallen en algunos entornos de Colab
render_monitor()
threading.Thread(target=update_monitor, daemon=True).start()
print("✅ Monitor activado (VRAM + disco + tiempo, cada 5 s)")

## 🧮 Celda 7 — Validar entorno y elegir modelo

Detecta tu GPU (VRAM) y el disco libre, y **desbloquea solo las variantes del checkpoint que caben en tu VRAM**. El disco no bloquea: si va justo solo muestra un aviso ⚠️ (es una estimación, y no aplica si ya descargaste los modelos en esta sesión):

| Entorno | Variantes desbloqueadas (aprox.) |
|---|---|
| Colab gratuito (T4, ~15 GB VRAM) | GGUF hasta `Q4_K_S` (el por defecto) |
| Colab Pro (L4, ~22.5 GB VRAM) | GGUF hasta `Q6_K` |
| A100 (40 GB VRAM) | Todo, incluido el **fp8 original sin cuantizar** (29.2 GB) |

Elige en el desplegable; la Celda 8 usará tu selección automáticamente. Si te saltas esta celda, la Celda 8 usa su parámetro `GGUF_QUANT` como hasta ahora.

In [ ]:
# ─── Validar entorno (GPU / VRAM / disco) y elegir modelo ──────────────────
import os, shutil, subprocess

# Catálogo: (clave, tamaño del checkpoint GB, VRAM mínima estimada GB)
# Los requisitos son estimaciones: tamaño del archivo + margen para
# activaciones/VAEs. FP8_ORIGINAL = checkpoint sin cuantizar (sin parches GGUF).
MODEL_CATALOG = [
    ('Q3_K_S',       10.3, 11.5),
    ('Q3_K_M',       11.2, 12.5),
    ('Q4_0',         12.6, 14.0),
    ('Q4_K_S',       13.2, 14.5),
    ('Q4_K_M',       14.0, 15.5),
    ('Q5_K_S',       15.6, 17.0),
    ('Q5_K_M',       16.1, 17.5),
    ('Q6_K',         18.5, 20.0),
    ('Q8_0',         23.9, 25.5),
    ('FP8_ORIGINAL', 29.2, 31.0),
]
# Disco que falta por consumir DESPUÉS de la Celda 3 (deps ya instaladas):
# ComfyUI + custom nodes + text encoder + LoRAs + VAEs + upscaler (estimado).
BASE_DISK_GB = 48

def _model_label(key, size_gb):
    if key == 'FP8_ORIGINAL':
        return f'fp8 mixed original ({size_gb} GB, máxima calidad)'
    return f'GGUF {key} ({size_gb} GB)'

# GPU info: leer del env si Celda 1 ya la detectó (evita segunda llamada a nvidia-smi)
try:
    if 'GPU_NAME' in os.environ and 'GPU_VRAM_MB' in os.environ:
        gpu_name = os.environ['GPU_NAME']
        vram_gb = float(os.environ['GPU_VRAM_MB']) / 1024
    else:
        out = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
            capture_output=True, text=True, check=True).stdout.strip().splitlines()[0]
        gpu_name, vram_mb = (s.strip() for s in out.split(','))
        vram_gb = float(vram_mb) / 1024
except Exception:
    gpu_name, vram_gb = 'no detectada', 0.0

disk_free_gb = shutil.disk_usage('/content').free / 1024**3
print(f'🖥️  GPU: {gpu_name} — {vram_gb:.1f} GB VRAM')
print(f'💽 Disco libre: {disk_free_gb:.1f} GB')
print('   (la VRAM bloquea variantes; el disco solo avisa — es una estimación,')
print('    y si ya descargaste los modelos en esta sesión el aviso no aplica)\n')

if vram_gb <= 0:
    raise RuntimeError('❌ No se detectó GPU (nvidia-smi falló). '
                       'Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4). '
                       'Si ya está en GPU, puede que agotaras la cuota gratuita — espera unas horas.')

unlocked, rows = [], []
for key, size_gb, vram_req in MODEL_CATALOG:
    need_disk = BASE_DISK_GB + size_gb
    ok_vram = vram_gb >= vram_req
    ok_disk = disk_free_gb >= need_disk
    if ok_vram:
        unlocked.append(key)
        status = '✅ disponible' if ok_disk else f'⚠️  disco justo (necesita ~{need_disk:.0f} GB libres)'
    else:
        status = f'🔒 VRAM: necesita ~{vram_req:.0f} GB'
    rows.append((_model_label(key, size_gb), status))

w = max(len(r[0]) for r in rows)
for label, status in rows:
    print(f'  {label:<{w}}  {status}')

if not unlocked:
    raise RuntimeError(f'❌ Ninguna variante cabe en {vram_gb:.1f} GB de VRAM ({gpu_name}).')

# Por defecto: Q4_K_S si está desbloqueado (el probado en T4); si no, el mejor disponible
default = 'Q4_K_S' if 'Q4_K_S' in unlocked else unlocked[-1]
os.environ['LTX_MODEL_CHOICE'] = default

import ipywidgets as widgets
from IPython.display import display

_sizes = {k: s for k, s, _ in MODEL_CATALOG}
dd = widgets.Dropdown(
    options=[(_model_label(k, _sizes[k]), k) for k in unlocked],
    value=default,
    description='Modelo:',
    layout=widgets.Layout(width='420px'),
)
def _on_select(change):
    os.environ['LTX_MODEL_CHOICE'] = change['new']
    print(f'✅ Modelo seleccionado: {change["new"]}')
dd.observe(_on_select, names='value')
display(dd)
print(f'\n✅ Selección actual: {default} — cámbiala en el desplegable si quieres; '
      'la Celda 8 la usará automáticamente')

## 🚀 Celda 8 — Parchear y Lanzar la App

Aplica los parches en memoria (el `app.py` descargado no se modifica) y lanza la app como **proceso independiente del kernel** — la UI de Gradio ya **no se incrusta en la celda**:
- La celda sigue los logs solo hasta que aparece el **enlace público** `https://...gradio.live` y entonces **termina**; la app sigue corriendo en segundo plano.
- **Re-ejecutarla es seguro**: detiene la instancia anterior antes de relanzar (antes, cada re-ejecución cargaba otra copia de los modelos en la RAM del kernel → OOM).
- **Logs completos** en `/content/app.log` → desde cualquier celda: `!tail -f /content/app.log` (en vivo) o `!tail -n 80 /content/app.log`.
- Usa el **modelo elegido en la Celda 7** (`LTX_MODEL_CHOICE`). Si omitiste la Celda 7, se usa `Q4_K_S` como fallback (cabe en T4). Con `FP8_ORIGINAL` se omiten los parches GGUF y se usa el checkpoint original de 29.2 GB.
- Text encoder fp8 (13.2 GB) o fp4 (9.4 GB) si activas `USE_FP4_TEXT_ENCODER`
- `DISABLE_ENHANCER` (activado por defecto): omite el **prompt enhancer** — su modelo sulphur (~13 GB) + llama.cpp **no caben** en el disco del Colab gratuito.
- **Checkboxes `SLOT1`…`SLOT6`**: cada LoRA original de slot se descarga solo si su checkbox está marcado (por defecto, todas). Desmarcar una ahorra su tamaño en disco y su dropdown en la UI arranca en `(none)`.
- LoRAs custom de la Celda 5: instala **solo las marcadas** en sus checkboxes y retira las desmarcadas — solo las marcadas aparecen en los **dropdowns de cada slot**.

> La primera ejecución descarga los modelos (~45-85 GB según la variante, 10-25 min con `hf_transfer`).
> Cuando la celda imprima `✅ APP LISTA → https://...gradio.live` → abre ese enlace.

In [ ]:
# ─── Parches GGUF + lanzamiento ────────────────────────────────────────────
USE_FP4_TEXT_ENCODER = False  #@param {type:"boolean"}
DISABLE_ENHANCER = True  #@param {type:"boolean"}

# Modelo: tomado de la Celda 7 (LTX_MODEL_CHOICE). Si omitiste la Celda 7,
# se usa Q4_K_S como fallback (cabe en la T4 gratuita). Para cambiarlo sin
# ejecutar la Celda 7: !export LTX_MODEL_CHOICE=Q5_K_S (en otra celda).
GGUF_QUANT = "Q4_K_S"  # fallback; la Celda 7 lo sobreescribe vía LTX_MODEL_CHOICE

# LoRAs ORIGINALES de los slots — desmarca las que NO quieras descargar
# (ahorra su tamaño en disco; el dropdown de ese slot arrancará en "(none)"
# y solo ofrecerá tus LoRAs custom de la Celda 5):
SLOT1_HARDCUT = True        #@param {type:"boolean"}
SLOT2_SYNTH = True          #@param {type:"boolean"}
SLOT3_PLORA = True          #@param {type:"boolean"}
SLOT4_OMNINFT_BF16 = True   #@param {type:"boolean"}
SLOT5_BETTER_MOTION = True  #@param {type:"boolean"}
SLOT6_PHYSICS_V2 = True     #@param {type:"boolean"}

# La app ya NO se ejecuta dentro del kernel: se lanza como subproceso, así que
# ni gradio ni el mock de 'spaces' se cargan aquí (el mock vive en _colab_run.py).
import os, re, signal, subprocess, sys, time

MODEL_CHOICE = os.environ.get('LTX_MODEL_CHOICE', GGUF_QUANT)
USE_GGUF = MODEL_CHOICE != 'FP8_ORIGINAL'
if USE_GGUF:
    GGUF_QUANT = MODEL_CHOICE
if 'LTX_MODEL_CHOICE' in os.environ:
    print(f'🧩 Modelo: {MODEL_CHOICE}  ← tomado de la Celda 7')
else:
    print(f'⚠️  Modelo: {MODEL_CHOICE} (fallback — ejecuta la Celda 7 para validar tu entorno y elegir modelo)')

# Enhancer de prompts: desactivado por defecto en Colab — su modelo (sulphur
# q8, ~13 GB) + llama.cpp NO caben en el disco del Colab gratuito (~78 GB).
os.environ['DISABLE_ENHANCER'] = '1' if DISABLE_ENHANCER else ''
print('🚫 Prompt enhancer desactivado (ahorra ~14 GB de disco)' if DISABLE_ENHANCER
      else '✨ Prompt enhancer activado (necesita ~14 GB extra de disco)')

# Slots con la LoRA original desmarcada → la app no la descarga y su dropdown
# arranca en "(none)" (app_space.py lee SKIP_SLOT_LORAS al arrancar)
_skip_slots = [s for s, keep in {
    'slot1': SLOT1_HARDCUT, 'slot2': SLOT2_SYNTH, 'slot3': SLOT3_PLORA,
    'slot4': SLOT4_OMNINFT_BF16, 'slot5': SLOT5_BETTER_MOTION,
    'slot6': SLOT6_PHYSICS_V2,
}.items() if not keep]
os.environ['SKIP_SLOT_LORAS'] = ','.join(_skip_slots)
print('🎚️ LoRAs originales de slots omitidas: '
      + (', '.join(_skip_slots) if _skip_slots else 'ninguna (todas se descargan)'))

SPACE_DIR = '/content/LTX-10Eros'
GGUF_FILE = f'10Eros_v1-{GGUF_QUANT}.gguf'

app_py = os.path.join(SPACE_DIR, 'app.py')
with open(app_py, encoding='utf-8') as f:
    source = f.read()

_failed = []
def patch(old, new, label):
    global source
    if old not in source:
        print(f"⚠️  parche '{label}' NO aplicado (¿cambió el código del Space?)")
        _failed.append(label)
        return
    source = source.replace(old, new)
    print(f"✅ parche '{label}' aplicado")

def gpatch(old, new, label):
    """patch() solo en modo GGUF; con FP8_ORIGINAL el checkpoint original se usa tal cual."""
    if USE_GGUF:
        patch(old, new, label)
    else:
        print(f"⏭️  parche '{label}' omitido (fp8 original)")

# ── P1: checkpoint fp8 → GGUF + VAEs + text projection en DOWNLOADS ──
gpatch('''    {
        "repo": "TenStrip/LTX2.3-10Eros",
        "file": "10Eros_v1-fp8mixed_learned.safetensors",
        "dest": MODELS / "checkpoints" / "10Eros_v1-fp8mixed_learned.safetensors",
        "label": "main checkpoint",
    },''',
f'''    {{
        "repo": "vantagewithai/LTX2.3-10Eros-GGUF",
        "file": "{GGUF_FILE}",
        "dest": MODELS / "diffusion_models" / "{GGUF_FILE}",
        "label": "main checkpoint (GGUF {GGUF_QUANT})",
    }},
    {{
        "repo": "Kijai/LTX2.3_comfy",
        "file": "vae/LTX23_video_vae_bf16.safetensors",
        "dest": MODELS / "vae" / "LTX23_video_vae_bf16_KJ.safetensors",
        "label": "video VAE (Kijai)",
    }},
    {{
        "repo": "Kijai/LTX2.3_comfy",
        "file": "vae/LTX23_audio_vae_bf16.safetensors",
        "dest": MODELS / "vae" / "LTX23_audio_vae_bf16_KJ.safetensors",
        "label": "audio VAE (Kijai)",
    }},
    {{
        "repo": "Kijai/LTX2.3_comfy",
        "file": "text_encoders/ltx-2.3_text_projection_bf16.safetensors",
        "dest": MODELS / "text_encoders" / "ltx-2.3_text_projection_bf16.safetensors",
        "label": "text projection (Kijai)",
    }},''',
'P1: DOWNLOADS checkpoint→GGUF + VAEs')

# ── P2: workflow MSR (runexx) — nodo 59 UNETLoader → UnetLoaderGGUF ──
gpatch('''            node["type"] = "CheckpointLoaderSimple"
            node["widgets_values"] = ["10Eros_v1-fp8mixed_learned.safetensors"]''',
f'''            node["type"] = "UnetLoaderGGUF"
            node["widgets_values"] = ["{GGUF_FILE}"]''',
'P2: runexx nodo 59 → UnetLoaderGGUF')

# ── P3: runexx nodo 57 — restaurar DualCLIPLoader original (gemma + text projection) ──
gpatch('''        elif nid == RUNEXX_NODE_CLIP_LOADER:
            node["type"] = "LTXAVTextEncoderLoader"
            node["widgets_values"] = [
                "gemma_3_12B_it_fp8_scaled.safetensors",
                "10Eros_v1-fp8mixed_learned.safetensors",
                "default",
            ]''',
'''        elif nid == RUNEXX_NODE_CLIP_LOADER:
            node["type"] = "DualCLIPLoader"
            node["widgets_values"] = [
                "gemma_3_12B_it_fp8_scaled.safetensors",
                "ltx-2.3_text_projection_bf16.safetensors",
                "ltxv",
                "default",
            ]''',
'P3: runexx nodo 57 → DualCLIPLoader')

# ── P4: runexx nodo 53 — restaurar VAELoaderKJ original (audio VAE) ──
gpatch('''        elif nid == RUNEXX_NODE_VAE_AUDIO:
            node["type"] = "LTXVAudioVAELoader"
            node["widgets_values"] = ["10Eros_v1-fp8mixed_learned.safetensors"]''',
'''        elif nid == RUNEXX_NODE_VAE_AUDIO:
            node["type"] = "VAELoaderKJ"
            node["widgets_values"] = ["LTX23_audio_vae_bf16_KJ.safetensors", "main_device", "bf16"]''',
'P4: runexx nodo 53 → VAELoaderKJ')

# ── P5: runexx — mantener el VAELoader de video original (nodo 56) ──
gpatch('''    skip_ids = {
        RUNEXX_NODE_VAE_VIDEO,
        RUNEXX_NODE_VAE_TINY,''',
'''    skip_ids = {
        RUNEXX_NODE_VAE_TINY,''',
'P5: runexx no saltar nodo 56 (video VAE)')

gpatch('''        (RUNEXX_NODE_VAE_VIDEO, 0): [str(RUNEXX_NODE_UNET_LOADER), 2],
''', '', 'P6: runexx quitar rewire VAE→checkpoint')

# ── P7: funciones inyectadas (patch del workflow default + instalador de loras custom) ──
_PRELUDE = '''
def _colab_install_custom_loras():
    \'\'\'Sincroniza el catálogo con los checkboxes de la Celda 5 (selected.json):
    instala las LoRAs custom marcadas y retira las desmarcadas.\'\'\'
    src_dir = pathlib.Path("/content/custom_loras")
    if not src_dir.exists():
        return
    selected = None  # sin selected.json (Celda 5 vieja/no ejecutada) -> todas
    sel_path = src_dir / "selected.json"
    if sel_path.exists():
        try:
            selected = set(json.loads(sel_path.read_text()))
        except Exception:
            selected = None
    dest_dir = MODELS / "loras" / "ltx23" / "custom"
    dest_dir.mkdir(parents=True, exist_ok=True)
    for f in src_dir.glob("*.safetensors"):
        dest = dest_dir / f.name
        if selected is not None and f.name not in selected:
            if dest.exists():
                dest.unlink()
                print(f"[colab] custom lora retirada (desmarcada): {f.name}", flush=True)
            continue
        if not dest.exists():
            try:
                os.link(f, dest)
            except OSError:
                shutil.copy2(f, dest)
            print(f"[colab] custom lora instalada: {f.name}", flush=True)
    # manifest con metadatos de CivitAI -> nombres legibles en los dropdowns
    man = src_dir / "mapping.json"
    if man.exists():
        shutil.copy2(man, dest_dir / "manifest.json")


def _colab_gguf_patch_default(api):
    \'\'\'Reemplaza CheckpointLoaderSimple (fp8 29GB) por UnetLoaderGGUF +
    VAEs sueltos + DualCLIPLoader (config probada del workflow runexx).\'\'\'
    api["646"] = {"class_type": "UnetLoaderGGUF",
                  "inputs": {"unet_name": "__COLAB_GGUF_FILE__"}}
    api["64600"] = {"class_type": "VAELoader",
                    "inputs": {"vae_name": "LTX23_video_vae_bf16_KJ.safetensors"}}
    api["617"] = {"class_type": "VAELoaderKJ",
                  "inputs": {"vae_name": "LTX23_audio_vae_bf16_KJ.safetensors",
                             "device": "main_device", "weight_dtype": "bf16"}}
    api["616"] = {"class_type": "DualCLIPLoader",
                  "inputs": {"clip_name1": "gemma_3_12B_it_fp8_scaled.safetensors",
                             "clip_name2": "ltx-2.3_text_projection_bf16.safetensors",
                             "type": "ltxv", "device": "default"}}
    for _node in api.values():
        _inputs = _node.get("inputs", {})
        for _k, _v in list(_inputs.items()):
            if (isinstance(_v, list) and len(_v) == 2
                    and str(_v[0]) == "646" and str(_v[1]) == "2"):
                _inputs[_k] = ["64600", 0]
    print("[colab] workflow default parcheado a GGUF", flush=True)


'''
patch('def _ensure_comfy() -> None:',
      _PRELUDE + 'def _ensure_comfy() -> None:',
      'P7: inyectar funciones colab')
source = source.replace('__COLAB_GGUF_FILE__', GGUF_FILE)

# ── P8: aplicar patch GGUF al workflow default tras la conversión ──
gpatch('        _workflow_cache["default"] = _convert_workflow(path)',
'''        _workflow_cache["default"] = _convert_workflow(path)
        _colab_gguf_patch_default(_workflow_cache["default"])''',
'P8: patch default workflow')

# ── P9: instalar loras custom al preparar ComfyUI ──
# Sincroniza el catálogo con los checkboxes de la Celda 5 (selected.json):
# instala las marcadas y retira las desmarcadas. Las originales se descargan
# según los checkboxes de arriba; la elección por slot va en los dropdowns.
patch('''
    _comfy_ready = True
''',
'''
    _colab_install_custom_loras()
    _comfy_ready = True
''',
'P9: hook instalador loras custom')

# ── P10 (opcional): text encoder fp4 (9.4 GB en vez de 13.2 GB) ──
if USE_FP4_TEXT_ENCODER:
    n = source.count('gemma_3_12B_it_fp8_scaled.safetensors')
    source = source.replace('gemma_3_12B_it_fp8_scaled.safetensors',
                            'gemma_3_12B_it_fp4_mixed.safetensors')
    print(f'✅ P10: text encoder → fp4_mixed ({n} referencias)')

if _failed:
    print(f'\n⚠️  {len(_failed)} parches fallaron: {_failed}')
    print('   La app puede intentar descargar el checkpoint fp8 de 29 GB. Revisa antes de continuar.')
else:
    print('\n✅ Todos los parches aplicados correctamente')

# ── Lanzar la app como PROCESO INDEPENDIENTE del kernel ────────────────────
# Antes la celda ejecutaba la app con exec(): eso incrustaba la UI de Gradio
# en la celda y, al re-ejecutarla, cargaba una SEGUNDA copia de los modelos en
# la RAM del kernel (causa del OOM). Ahora la app corre en un subproceso con
# logs en /content/app.log; la celda solo los sigue hasta ver el enlace.
APP_PATCHED = os.path.join(SPACE_DIR, 'app_colab.py')
LAUNCHER = os.path.join(SPACE_DIR, '_colab_run.py')
LOG_PATH = '/content/app.log'
PID_PATH = '/content/app.pid'

with open(APP_PATCHED, 'w', encoding='utf-8') as f:
    f.write(source)  # el app.py descargado NO se modifica

# Lanzador del subproceso: mock de 'spaces' + share=True (hereda los env vars
# del kernel: tokens, LTX_MODEL_CHOICE, DISABLE_ENHANCER, SKIP_SLOT_LORAS).
_LAUNCHER_CODE = ('''\
import sys, types

try:
    import spaces  # noqa: F401
except ModuleNotFoundError:
    _mock = types.ModuleType('spaces')
    def _gpu_decorator(*args, **kwargs):
        if len(args) == 1 and callable(args[0]) and not kwargs:
            return args[0]
        def _decorator(fn):
            return fn
        return _decorator
    _mock.GPU = _gpu_decorator
    _mock.ZeroGPU = _gpu_decorator
    sys.modules['spaces'] = _mock

import gradio as gr
_orig_launch = gr.Blocks.launch
def _share_launch(self, *args, **kwargs):
    kwargs['share'] = True
    print('[colab] iniciando servidor Gradio con enlace publico...', flush=True)
    return _orig_launch(self, *args, **kwargs)
gr.Blocks.launch = _share_launch

_app = '__APP_PATCHED__'
with open(_app, encoding='utf-8') as f:
    _src = f.read()
exec(compile(_src, _app, 'exec'), {'__file__': _app, '__name__': '__main__'})
'''.replace('__APP_PATCHED__', APP_PATCHED))
with open(LAUNCHER, 'w', encoding='utf-8') as f:
    f.write(_LAUNCHER_CODE)

# Detener la instancia anterior si existe (dos copias = OOM seguro en Colab)
if os.path.exists(PID_PATH):
    try:
        _old = int(open(PID_PATH).read().strip())
        os.killpg(_old, signal.SIGTERM)
        for _ in range(25):  # esperar hasta 5 s en vez de sleep fijo
            if not os.path.exists(f'/proc/{_old}'):
                break
            time.sleep(0.2)
        print(f'🛑 Instancia anterior detenida (pid {_old})')
    except (ValueError, ProcessLookupError, PermissionError):
        pass
subprocess.run(['pkill', '-9', '-f', '_colab_run.py'], capture_output=True)  # huérfanos
time.sleep(0.5)  # pausa mínima para liberar el puerto

_model_desc = f'GGUF {GGUF_QUANT}' if USE_GGUF else 'fp8 mixed original — sin parches GGUF'
_log_f = open(LOG_PATH, 'w', encoding='utf-8')  # se trunca en cada lanzamiento
_proc = subprocess.Popen(
    [sys.executable, '-u', LAUNCHER],
    cwd=SPACE_DIR, stdout=_log_f, stderr=subprocess.STDOUT,
    start_new_session=True,  # grupo propio: al matarlo caen también sus hijos
)
_log_f.close()
open(PID_PATH, 'w').write(str(_proc.pid))
print(f'\n▶️  App lanzada en segundo plano ({_model_desc}) — pid {_proc.pid}')
print(f'📜 Logs completos en {LOG_PATH} → en otra celda: !tail -f {LOG_PATH}')
print('📦 Primera vez: descarga de modelos ~45-85 GB según el modelo (10-25 min)')
print('⏳ Siguiendo los logs hasta que aparezca el enlace público...\n')

# Seguir el log SOLO hasta ver el enlace de Gradio; después la celda termina
# y la app sigue corriendo aunque ejecutes otras celdas.
_url, _pos, _buf = None, 0, ''
while _url is None:
    _alive = _proc.poll() is None
    with open(LOG_PATH, encoding='utf-8', errors='replace') as f:
        f.seek(_pos)
        _chunk = f.read()
        _pos = f.tell()
    if _chunk:
        print(_chunk, end='', flush=True)
        _buf = (_buf + _chunk)[-4000:]
        _m = re.search(r'https://[\w.-]+\.gradio\.live', _buf)
        if _m:
            _url = _m.group(0)
    if _url is None:
        if not _alive:
            break
        time.sleep(2)

if _url:
    print('\n' + '═' * 62)
    print(f'✅ APP LISTA → {_url}')
    print('═' * 62)
    print('La celda terminó; la app sigue corriendo en segundo plano.')
    print(f'   • Ver logs en vivo: !tail -f {LOG_PATH}')
    print('   • Detener la app:   Celda 10 (o re-ejecuta esta celda para reiniciarla)')
else:
    print('\n' + '═' * 62)
    print(f'❌ La app terminó (código {_proc.returncode}) antes de publicar el enlace.')
    print(f'   Diagnóstico: !tail -n 80 {LOG_PATH}')

## 🧹 Celda 9 — Limpieza profunda de disco (opcional)
> Ejecútala **después** de que la app haya arrancado al menos una vez (detén la Celda 8 primero). Borra los `.git` de ComfyUI/nodos y cachés residuales — los modelos no se tocan.

In [ ]:
# ─── Limpieza profunda (no toca modelos) ───────────────────────────────────
import subprocess, shutil, pathlib, os, signal

before = shutil.disk_usage('/content').free

# Verificar si la app sigue corriendo — borrar /tmp/gradio con la app activa
# elimina archivos de outputs en curso y puede romper la sesión de Gradio.
_app_running = False
if os.path.exists('/content/app.pid'):
    try:
        _pid = int(open('/content/app.pid').read().strip())
        os.kill(_pid, 0)  # señal 0 = solo verificar existencia del proceso
        _app_running = True
        print(f'⚠️  La app sigue corriendo (pid {_pid}).')
        print('   /tmp/gradio NO se borrará para no interrumpir sesiones activas.')
        print('   Detener la app primero (Celda 10) si quieres limpiarlos también.\n')
    except (ValueError, ProcessLookupError, PermissionError):
        pass

cmds = [
    'pip cache purge',
    'apt-get clean',
    'rm -rf /root/.cache/pip',
    # .git de ComfyUI y custom nodes (la app no los re-clona si el dir existe)
    'find /content/LTX-10Eros/ComfyUI -name .git -type d -prune -exec rm -rf {} +',
    # restos del prompt enhancer (clon de llama.cpp, pesos sulphur, caché HF del binario)
    'rm -rf /content/LTX-10Eros/llama.cpp /content/LTX-10Eros/sulphur_enhancer',
    'rm -rf /content/LTX-10Eros/llama-server-cached /content/LTX-10Eros/llama-server-libs',
    'rm -rf /root/.cache/huggingface/hub/models--signsur4739379373--ltx-dependencies',
    'rm -rf /root/.cache/huggingface/hub/models--SulphurAI--Sulphur-2-base',
]
# Temporales de Gradio: solo si la app no está corriendo
if not _app_running:
    cmds.append('rm -rf /tmp/gradio /tmp/tmp*')
for c in cmds:
    print(f'$ {c}')
    subprocess.run(c, shell=True)

after = shutil.disk_usage('/content').free
print(f'\n🧹 Liberados {(after-before)/1024**3:.2f} GB')
print(f'💽 Disco libre: {after/1024**3:.1f} GB')

# Top de carpetas más pesadas para diagnóstico
print('\n📂 Carpetas más pesadas:')
subprocess.run('du -h --max-depth=2 /content/LTX-10Eros 2>/dev/null | sort -rh | head -15', shell=True)

## 🛑 Celda 10 — Detener la app, el monitor y liberar GPU (opcional)

In [ ]:
# ─── Detener app, monitor y liberar memoria GPU ────────────────────────────
import torch, gc, os, signal

# Detener la app (subproceso lanzado por la Celda 8)
if os.path.exists('/content/app.pid'):
    try:
        _pid = int(open('/content/app.pid').read().strip())
        os.killpg(_pid, signal.SIGTERM)
        print(f'🛑 App detenida (pid {_pid})')
    except (ValueError, ProcessLookupError, PermissionError):
        print('ℹ️  La app ya no estaba corriendo')
    os.remove('/content/app.pid')

try:
    _monitor_running = False
    print("⏹️  Monitor detenido")
except NameError:
    pass

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    used  = torch.cuda.memory_allocated() / 1024**2
    total = torch.cuda.get_device_properties(0).total_memory / 1024**2
    print(f"🧹 Caché CUDA limpiada — VRAM: {used:.0f} / {total:.0f} MB")

print("\n✅ Listo. Re-ejecuta la Celda 8 para reiniciar la app.")

---
## ℹ️ Notas importantes

| Tema | Detalle |
|------|--------|
| **Cuantización** | `Q4_K_S` (13.2 GB) cabe en la VRAM de la T4. `Q5_K_S`/`Q6_K` mejoran calidad pero requieren GPU con más VRAM (Colab Pro: L4/A100). La **Celda 7** valida el entorno y desbloquea solo las variantes ejecutables (incluido el fp8 original en A100) |
| **Text encoder fp4** | Activa `USE_FP4_TEXT_ENCODER` si vas corto de disco o RAM (−3.8 GB, leve pérdida de calidad de prompt) |
| **LoRA destilada** | Se mantiene (7.6 GB): es **necesaria** para el sampling de pocos pasos del workflow principal |
| **LoRAs custom** | Las originales de cada slot se descargan según los checkboxes de la Celda 8 (por defecto, todas). Las de la Celda 5 tienen sus **propios checkboxes**: solo las marcadas se instalan y aparecen en el **dropdown** de cada slot (original / custom / `(none)`); el slider del slot controla el archivo elegido |
| **Presets** | Ajustan las *fuerzas* de los sliders, no el archivo del dropdown — la elección de LoRA por slot sobrevive a los cambios de preset |
| **Token HF** | No acelera descargas; sirve para repos gated y rate-limits. `hf_transfer` (Celda 2) sí acelera |
| **Persistencia** | `/content` se borra al reiniciar el runtime — habrá que re-descargar modelos |
| **Fuente del código** | La Celda 3 descarga `app.py` y el workflow MSR del repositorio de GitHub (`aiRabbit0/SpaceLTX-GoogleColab`), no del Space. Los modelos se descargan de repos públicos de HF, independientes del Space |
| **RAM / kernel** | La app corre en un **subproceso** (la Celda 8 no la carga en el kernel): re-ejecutar la Celda 8 detiene la instancia anterior y relanza sin duplicar RAM. Si aun así se agota, reinicia el kernel y re-ejecuta la Celda 8 — no hace falta repetir las descargas. Logs: `/content/app.log` |

### ✏️ Cómo personalizar la app (para desarrolladores)
El código que se ejecuta es `app_space.py` del repositorio de GitHub. El ciclo de cambios:
1. Haz un **fork** del repo y edita `app_space.py` (o `workflow_runexx.json`)
2. `git push` a tu fork
3. En esta copia del notebook: cambia `CODE_RAW` (Celda 3) a tu fork, re-ejecuta la **Celda 3** y luego la **Celda 8**

⚠️ Si el cambio toca alguno de los strings que la Celda 8 usa como objetivo de parche (el bloque del checkpoint en `DOWNLOADS`, los loaders del workflow runexx, `_ensure_comfy`, `_comfy_ready`...), hay que actualizar también el `old` correspondiente en la Celda 8 — si no, aparecerá `⚠️ parche ... NO aplicado`.